## 03: Modeling
We aim to predict maize line GCA, or General Combining Ability, which is the average yield across environments from genomic + environmental + phenotypic data.

Our Pipeline involves a baseline of location-year means, then Ridge regression for genomic prediction, then K-fold CV, then uncertainty quantification, and finally ranked line advancement list.

Flip `SAMPLE_MODE = False` for the full run.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

SAMPLE_MODE = False  #flip to False for full run

DATA_PATH   = '../data/processed/merged_sample.csv' if SAMPLE_MODE \
              else '../data/processed/merged_full.csv'
OUTPUT_PATH = '../outputs/line_rankings_sample.csv' if SAMPLE_MODE \
              else '../outputs/line_rankings_full.csv'

import os; os.makedirs('../outputs', exist_ok=True)
print('SAMPLE_MODE:', SAMPLE_MODE)


SAMPLE_MODE: False


### 1. Load merged data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')

#drop rows with no yield target
df = df.dropna(subset=['YLD_BE']).reset_index(drop=True)
print(f'After dropping missing YLD_BE: {df.shape}')


Loaded: (532448, 3019)
After dropping missing YLD_BE: (511642, 3019)


### 2. Define feature sets
Three feature groups, used progressively:
- **Baseline**: location + year mean yield (no ML needed)
- **Pheno-only**: observable trait features (MST, PHT, etc.)
- **Full**: pheno + env (climate/soil) + genomic (SNP markers)

In [3]:
TARGET = 'YLD_BE'
ID_COLS = ['LINE_UNIQUE_ID', 'CLUSTER_ID', 'POP_NUM', 'LINE_NUM',
           'YEAR', 'LOC', 'LINE', 'CROSS', 'GERMPLASM_ID_TESTER',
           'SET', 'LONGITUDE', 'LATITUDE']

PHENO_FEATS = [c for c in ['MST', 'PHT', 'EHT', 'TWT', 'RTLP', 'STLP', 'ERM']
               if c in df.columns]
ENV_FEATS   = [c for c in df.columns if c.startswith('X')]
SNP_FEATS   = [c for c in df.columns if c.startswith('M')]

print(f'Pheno features : {len(PHENO_FEATS)}')
print(f'Env features   : {len(ENV_FEATS)}')
print(f'SNP features   : {len(SNP_FEATS)}')
print(f'Total features : {len(PHENO_FEATS)+len(ENV_FEATS)+len(SNP_FEATS)}')


Pheno features : 7
Env features   : 42
SNP features   : 2912
Total features : 2961


### 3. Baseline model: location-year mean yield
Required by the hackathon rubric. Predicts each observation as the mean yield of its (LOC, YEAR) group. This captures environmental effects only with no genomic information.

In [4]:
loc_year_means = df.groupby(['LOC', 'YEAR'])[TARGET].transform('mean')
baseline_resid  = df[TARGET] - loc_year_means

baseline_r2  = r2_score(df[TARGET], loc_year_means)
baseline_rmse = np.sqrt(((df[TARGET] - loc_year_means)**2).mean())

print('=== Baseline: LOC+YEAR mean ===')
print(f'  R²   : {baseline_r2:.3f}')
print(f'  RMSE : {baseline_rmse:.2f} bu/acre')
print(f'  (Note: in sample mode, all rows share a small set of LOC+YEAR groups')
print(f'   so this baseline is artificially strong on the same data)')


=== Baseline: LOC+YEAR mean ===
  R²   : 0.682
  RMSE : 20.17 bu/acre
  (Note: in sample mode, all rows share a small set of LOC+YEAR groups
   so this baseline is artificially strong on the same data)


### 4. Ridge regression (genomic prediction model)
Ridge regression with L2 penalty is computationally equivalent to G-BLUP, the industry standard for genomic selection. It handles the wide-matrix problem (many more SNP columns than rows) by shrinking all marker effects toward zero rather than zeroing any out.

In [5]:
FULL_FEATS = PHENO_FEATS + ENV_FEATS + SNP_FEATS
X = df[FULL_FEATS]
y = df[TARGET]

# Pipeline: impute NaN with column mean -> scale -> ridge
# Mean imputation for SNPs is neutral (mean SNP value ~0 = heterozygous)
model = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('ridge',  RidgeCV(alphas=[0.1, 1, 10, 100, 1000], cv=5)),
])

print(f'Feature matrix shape: {X.shape}')
print('Pipeline: impute -> scale -> RidgeCV (auto tunes alpha via inner 5-fold)')


Feature matrix shape: (511642, 2961)
Pipeline: impute -> scale -> RidgeCV (auto tunes alpha via inner 5-fold)


### 5. Cross-validation
We cross-validate at the **line level**, not the observation level. The same line can appear across multiple location-years; if we naively K-fold by row, the model sees near-identical rows in both train and test and CV accuracy is inflated. Holding out all observations for a line at once is the correct breeding program structure.

In [ ]:
lines = df['LINE_UNIQUE_ID'].values
unique_lines = np.unique(lines)
n_splits = min(5, len(unique_lines))

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_pred_cv = np.full(len(y), np.nan)

for fold, (train_idx, test_idx) in enumerate(kf.split(unique_lines)):
    train_lines = unique_lines[train_idx]
    test_lines  = unique_lines[test_idx]
    train_mask  = np.isin(lines, train_lines)
    test_mask   = np.isin(lines, test_lines)
    model.fit(X[train_mask], y[train_mask])
    y_pred_cv[test_mask] = model.predict(X[test_mask])

valid = ~np.isnan(y_pred_cv)
cv_r2   = r2_score(y[valid], y_pred_cv[valid])
cv_rmse = np.sqrt(((y[valid] - y_pred_cv[valid])**2).mean())

print(f'=== Ridge CV ({n_splits}-fold, line-level) ===')
print(f'  R²   : {cv_r2:.3f}')
print(f'  RMSE : {cv_rmse:.2f} bu/acre')
print(f'  vs baseline R²: {baseline_r2:.3f}')
print()
print('Note: with sample data (~97 rows, 1 location-year group, no env features),')
print('CV scores reflect pheno+SNP only and will improve substantially on full data.')


### 6. Uncertainty quantification
Bootstrap the CV residuals to produce 90% prediction intervals per line. High-uncertainty lines should be flagged as risky advancement candidates regardless of their point estimate.

In [ ]:
#fit final model on all data
model.fit(X, y)
y_pred_final = model.predict(X)

#bootstrap prediction intervals from CV residuals
cv_residuals = y.values[valid] - y_pred_cv[valid]
np.random.seed(42)
n_boot = 500
boot_preds = np.array([
    y_pred_final + np.random.choice(cv_residuals, size=len(y_pred_final), replace=True)
    for _ in range(n_boot)
])

pred_lo = np.percentile(boot_preds, 5,  axis=0)
pred_hi = np.percentile(boot_preds, 95, axis=0)
pred_sd = boot_preds.std(axis=0)

print(f'Mean 90% PI width: {(pred_hi - pred_lo).mean():.2f} bu/acre')


### 7. Line advancement rankings
Aggregate predictions to line level (GCA estimate = mean predicted yield across all environments a line was tested in). Rank by point estimate; flag lines whose lower prediction bound is below the population median as high-risk.

In [ ]:
results = df[['LINE_UNIQUE_ID', 'CLUSTER_ID', 'POP_NUM', 'LINE_NUM']].copy()
results['YLD_BE_obs']  = y.values
results['YLD_BE_pred'] = y_pred_final
results['pred_lo_90']  = pred_lo
results['pred_hi_90']  = pred_hi
results['pred_sd']     = pred_sd

#GCA is avg predicted performance across tested environments
gca = results.groupby('LINE_UNIQUE_ID').agg(
    n_envs        = ('YLD_BE_obs', 'count'),
    GCA_pred      = ('YLD_BE_pred', 'mean'),
    GCA_obs       = ('YLD_BE_obs', 'mean'),
    pred_sd_mean  = ('pred_sd', 'mean'),
    pred_lo_mean  = ('pred_lo_90', 'mean'),
    pred_hi_mean  = ('pred_hi_90', 'mean'),
).reset_index()

gca['rank']        = gca['GCA_pred'].rank(ascending=False).astype(int)
median_pred        = gca['pred_lo_mean'].median()
gca['high_risk']   = gca['pred_lo_mean'] < median_pred
gca['advance']     = (gca['rank'] <= 20) & ~gca['high_risk']

gca = gca.sort_values('rank')
print(f'Total lines: {len(gca)}')
print(f'Recommended to advance (top 20, low risk): {gca["advance"].sum()}')
print()
print('Top 10 lines:')
display_cols = ['LINE_UNIQUE_ID', 'rank', 'GCA_pred', 'GCA_obs', 'pred_lo_mean', 'pred_hi_mean', 'high_risk', 'advance']
gca[display_cols].head(10)


### 8. Save outputs

In [ ]:
gca.to_csv(OUTPUT_PATH, index=False)
print(f'Rankings saved -> {OUTPUT_PATH}')
print()
print('=== Model summary ===')
print(f'  Baseline RMSE : {baseline_rmse:.2f} bu/acre  (R² {baseline_r2:.3f})')
print(f'  Ridge CV RMSE : {cv_rmse:.2f} bu/acre  (R² {cv_r2:.3f})')
print(f'  Features used : {len(FULL_FEATS)} ({len(PHENO_FEATS)} pheno, {len(ENV_FEATS)} env, {len(SNP_FEATS)} SNP)')
print(f'  Lines ranked  : {len(gca)}')
print(f'  Advance list  : {gca["advance"].sum()} lines')


SAMPLE DATA INTERP:

The baseline $R^2$ 0.496 vs Ridge $R^2$ 0.128 looks crucial, but the baseline is evaluated within the sample, or mean of the group the row belongs to, computed on the same data. On the other hand, ridge is cross-validated out of sample. If you ran the baseline through the same line level holdout CV, its $R^2$ would collapse as holding out a line means you have no group mean for it. The comparison here is in the correct direction but the baseline number is somewhat misleading itself. On full data both numbers will be meaningful and Ridge will win.

The 57 bu/acre PI width is wide because the CV residuals are large, which is because the model is essentially predicting from 7 pheno traits only or all env features imputed to their mean which equals uninformative constants, and 65% of SNPs are NaN. This shrinks on full data where real env features and imputed SNPs are available.

The rankings themselves look reasonable. C1.1.11 predicted 198, observed 205. C1.1.81 predicted 180, observed 182. The model is tracking the actual ordering even with degraded features.